In [ ]:
#tensorflow => tăng cường ảnh vì dataset chỉ có 60 ko đủ để train => phải dùng tensorflow (nhân bản: dọc, cheo, lật ngang...) => để tạo ra nhiều từ dataset 60 ảnh ban đầu
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# 1. TIÊN XỬ LÝ DỮ LIỆU
# Thiết lập đường dẫn dữ liệu
train_dir = "/content/drive/MyDrive/FACE_RAP001_K51"
# Kích thước ảnh và kích thước lô
img_width, img_height = 200, 200
batch_size = 32

# Tăng cường dữ liệu dành cho huấn luyện mô hình
train_datagen = ImageDataGenerator(
    rescale=1.0/255, # Normalize pixel values
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2 # Use 20% of the data for validation
)

# Validation chỉ chuẩn hóa, không augmentation
val_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2
)

# Tải dữ liệu huấn luyện và validation
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset='training'
)

validation_generator = val_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset='validation'
)

# XÂY DỰNG MÔ HÌNH CNN
model = Sequential([
    Input(shape=(img_width, img_height, 3)), # Use Input layer for defining input shape
    Conv2D(32, (3,3),
           activation="relu"),

    MaxPooling2D(2,2),

    Conv2D(64, (3,3),
           activation="relu"),

    MaxPooling2D(2,2),

    Conv2D(128, (3,3),
           activation="relu"),

    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation="relu"),

    Dropout(0.5),  # Reduce overfitting

    Dense(26, activation="softmax")  # 5 categories (assuming 6 categories based on previous output)
])

# Biên dịch mô hình
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
# Tóm tắt cấu hình của mô hình
model.summary()

# HUẤN LUYỆN MÔ HÌNH CNN
epochs = 10
history = model.fit(train_generator,
                    epochs=epochs,
                    validation_data=validation_generator)

# ĐÁNH GIÁ KẾT QUẢ MÔ HÌNH
plt.plot(history.history['accuracy'], label="Kết quả huấn luyện")
plt.plot(history.history['val_accuracy'], label="Độ chính xác xác thực")
plt.xlabel("Số lần học'")
plt.ylabel("Độ chính xác")
plt.legend()
plt.show()

# Tải và tiền xử lý ảnh kiểm tra
from keras.utils import load_img
import numpy as np
path = "/content/drive/MyDrive/FACE_RAP001_K51/Đỗ Trần Nam Mỹ/1.png"
# Tiên đoán loại s
img = load_img(path, target_size=(img_width, img_height))
plt.imshow(img)
plt.show()
img = np.array(img)
img = img / 255.0
img = img.reshape(1, img_width, img_height, 3)
prediction=np.argmax(model.predict(img), axis=1)[0]
#Ánh xạ loại tới tên người
class_labels = {v: k for k, v in train_generator.class_indices.items()}
person_name = class_labels[prediction]
print(f"Người tiên đoán: {person_name}")